# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset ([source](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)) using the `mlcroissant` library. The notebook walks through the main phases of data analysis: data loading, overview, extraction, EDA, and visualization. All dataset entities are referenced by their `@id` as per Croissant schema best practices.

### Dataset Source
The dataset source is: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nAuthors (by @id):")
if hasattr(metadata, 'author'):
    for author in metadata.author:
        print(f"  - {author['@id']}")
print(f"\nDataset Identifier: {getattr(metadata, 'identifier', None)}")
print(f"License: {getattr(metadata, 'license', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all RecordSets by their @id
print("Available record sets (by @id and name):")
record_sets = []
for record_set in dataset.record_sets:
    record_sets.append(record_set['@id'])
    print(f"- @id: {record_set['@id']}")
    print(f"    Name: {record_set.get('name', '(no name)')}")
    if 'field' in record_set:
        print("    Fields:")
        for field in record_set['field']:
            print(f"      - @id: {field['@id']}, Name: {field.get('name', '(no name)')}, DataType: {field.get('dataType', '')}")


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All access is done using record set and field `@id` as listed above.

In [ ]:
# Store DataFrames for each discovered record set
dataframes = {}

# Some datasets may not have record sets; check and load if available
if not record_sets:
    print("No record sets found in the dataset (recordSets were empty). If your dataset is reference-only, skip the extraction and proceed to metadata analysis.")
else:
    for record_set_id in record_sets:
        print(f"\nLoading records for record set: {record_set_id}")
        recs = list(dataset.records(record_set=record_set_id))
        if len(recs) == 0:
            print(f"  [Warning] No records found for {record_set_id}!")
            continue
        df = pd.DataFrame(recs)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} rows with columns:", df.columns.tolist())
    # For demonstration, preview the first loaded record set
    if dataframes:
        first_rs_id = list(dataframes.keys())[0]
        print(f"\nPreview of record set (first 5 rows): {first_rs_id}")
        display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing to a sample record set: filtering, normalizing, grouping, and summarization. All references use field and record set `@id` values.

In [ ]:
# Example EDA on the first available record set (by @id)

import numpy as np
if not dataframes:
    print("No dataframes available for EDA. Please verify record set extraction above.")
else:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"DataFrame columns for record set {record_set_id}:")
    print(df.columns.tolist())

    # Pick a likely numeric field by @id (e.g., field containing 'log_likelihood' or 'coefficient')
    numeric_field_id = None
    group_field_id = None
    # Attempt to automatically select suitable field ids
    for col in df.columns:
        if ('log_likelihood' in col.lower()) or ('coef' in col.lower()) or ('value' in col.lower()):
            numeric_field_id = col
        elif ('ward' in col.lower()) or ('county' in col.lower()) or ('group' in col.lower()):
            group_field_id = col

    if numeric_field_id is None:
        print("No obvious numeric field found for demonstration. Please adjust field selection as needed.")
    else:
        # Convert to numeric (force errors to NaN)
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        threshold = np.nanmean(df[numeric_field_id])
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize filtered column
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - mean) / std
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by a categorical field if available
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df)

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if not dataframes:
    print("No data loaded for visualization.")
else:
    # Use previous EDA results
    if 'filtered_df' in locals() and numeric_field_id:
        plt.figure(figsize=(8, 4))
        plt.hist(filtered_df[numeric_field_id].dropna(), bins=30, alpha=0.7, color='dodgerblue')
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.grid(True, linestyle=':', linewidth=0.5)
        plt.show()

        if group_field_id and group_field_id in filtered_df:
            plt.figure(figsize=(8, 4))
            filtered_df.boxplot(column=numeric_field_id, by=group_field_id)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.suptitle("")
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.show()
    else:
        print("No numeric field available to plot.")

## 6. Conclusion

- This notebook demonstrated how to use the `mlcroissant` library to load, inspect, and analyze a structured dataset defined in Croissant JSON-LD format. 
- All dataset queries and transformations referenced entities via their schema `@id`, following FAIR best practices for robust data science.
- You can now proceed to domain-specific analysis on this dataset, investigating relationships and validating results against the underlying research questions for rangeland management in Northern Kenya.
